# 08. 모델 확정과 핵심인자 — 전체 11단계 중 8단계

작업하며 주고받은 글(내가 셀마다 적은 칸과 클로드의 답변)은 빼고 코드 · 결과 · 결정만 남긴 판이다. 진행 방식은 README 「진행 방식」에 있다.

**이 단계에서 한 일과 정한 것** — 결과를 보기 전에 정한 고르는 방법 둘(두 해(2024 · 2025년) 모두 더 잘 맞히고 나눠 봐도 꾸준히 나은 모델 · 2025년 결정계수가 가장 높은 모델)이 모두 그래디언트 부스팅을 골랐다. 모델에 넣은 다섯이 모두 LNG 발전량을 움직였고, 2023 · 2024년 자료에서 몇 시인가 → 무슨 날인가 → 기온 → 햇빛 → 몇 월인가 순서로 크게 움직였다. 1단계 짐작 여섯 줄과 맞춰 보니 맞은 것은 추위 하나였다.

### 말 뜻 — 표를 읽을 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 실제 | 그해(주로 2026년)에 실제로 발전한 양 · 실제 값입니다. 전력거래소 자료 그대로입니다 |
| 모델 예측 | 2023 ~ 2025년을 배운 모델이 그때의 날씨 · 시각 · 날 종류를 보고 예측한 양입니다. 9단계까지는 「평소라면」이라고 불렀습니다 |
| 실제 − 모델 예측 | 발전량 표의 숫자입니다. + 면 실제가 모델 예측보다 많았다, − 면 적었다는 뜻입니다 |
| 평소 해 | 2023 · 2024 · 2025년입니다. 「평소」는 이 세 해만 가리킵니다 |
| 평소 범위 | 평소 해에 모델 예측이 한 달 평균으로 가장 크게 틀린 크기(±2,177 MWh)입니다. 모델이 예측해 본 2024 · 2025년으로 냈습니다. 2026년의 차이가 이 안이면 「달랐다고 말할 근거가 없다」로 읽습니다 |
| 세 해 평균 | 2023 ~ 2025년 같은 달 값의 평균입니다. 모델 없이 계산한 것으로, 10단계 가격 표(2-가 · 2-라)에서만 씁니다 |

앞 단계 글에서 발전량을 두고 쓴 「평소보다 많았다 · 적었다」는 「모델 예측보다 많았다 · 적었다」로 읽습니다.

In [97]:
# 8단계 모델 확정과 핵심인자 · 준비 ①
import os                                      # 폴더와 파일 경로를 다루는 기본 모듈
import numpy as np                             # 숫자 계산
import pandas as pd                            # 표(데이터프레임)를 다루는 라이브러리
import matplotlib.pyplot as plt                # 그림을 그리는 라이브러리
import sklearn                                 # 판 번호를 적어 두려고 부른다
from IPython.display import display            # 표를 글자가 아니라 격자로 그린다
from sklearn.linear_model import LinearRegression                                   # 선형회귀 — 7단계와 같다
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor   # 트리 계열 둘 — 7단계와 같다
plt.rcParams["font.family"] = "AppleGothic"    # 맥에서 그림 속 한글이 깨지지 않게
plt.rcParams["axes.unicode_minus"] = False     # 그림의 마이너스 부호가 깨지지 않게
pd.set_option("display.max_columns", 50)       # 열이 많아도 생략하지 않고 다 보이게

ROOT = os.getcwd()                             # 지금 폴더(보통 notebooks/)에서 시작해서
while not os.path.isdir(os.path.join(ROOT, "data")) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)               # data 폴더가 보일 때까지 위로 = 프로젝트 루트
HOT = os.path.join(ROOT, "data", "hot")        # 5단계가 만들어 둔 파일이 있는 폴더

print("프로젝트 루트:", ROOT)                    # 경로가 맞는지만 확인한다
print("scikit-learn 판:", sklearn.__version__)   # 그래디언트 부스팅은 판이 다르면 소수점 아래 넷째 숫자가 달라질 수 있어 적어 둔다(7단계 준비 ①)

프로젝트 루트: /Users/kimchangju/vs코드 /개인 프로젝트/lng-project
scikit-learn 판: 1.9.0


In [98]:
# 8단계 모델 확정과 핵심인자 · 준비 ②
tbl = pd.read_csv(os.path.join(HOT, "features_2023_2024_2025.csv"), encoding="utf-8-sig")  # 5단계 「저장」 셀이 만든 표 — 6 · 7단계와 같은 파일
tbl["ts"] = pd.to_datetime(tbl["ts"])            # 글자로 저장된 시각을 다시 시각 자료형으로
tbl = tbl.set_index("ts").sort_index()           # 시각을 줄 이름으로 두고 시간 순서대로 놓는다
tbl.loc[tbl["year"] == 2025, "lng_mwh"] = np.nan   # 누수를 막는다 — 2025년 LNG는 읽자마자 비우고, 준비 ③ 하나만 다시 읽는다

전체 = tbl.dropna(subset=["3시간 전 기온"])          # 3시간 전 기온이 빈 9시각(2023-01-01 · 2024-03-01 · 2024-09-01의 0~2시)은 뺀다 — 6 · 7단계와 같다
y = 전체["lng_mwh"]                                # 맞힐 값 — 그 시각의 LNG 발전량(MWh). 2025년은 비어 있다

def 선형열(d):                                     # 선형회귀에 넣는 열 — 7단계 준비 ②와 같은 67열
    시각열 = pd.get_dummies(d["hour24"].astype(str), prefix="시", drop_first=True).astype(float)   # 1~23시 0/1 열 — 0시가 기준
    월열 = pd.get_dummies(d["month"].astype(str), prefix="월", drop_first=True).astype(float)      # 2~12월 0/1 열 — 1월이 기준
    날열 = pd.get_dummies(d["날 종류"], prefix="날").astype(float).drop(columns="날_월")            # 화~금·토·일·공휴일·명절 — 월요일이 기준
    쉬는날 = d["날 종류"].isin(["토", "일", "공휴일", "명절"]).astype(float)                         # 쉬는 날이면 1
    쉬는날시간대 = (시각열 - 1 / 24).mul(쉬는날, axis=0)                                            # 5단계 물음 7 — 시간대 열의 평균(1/24)을 빼고 쉬는 날과 곱한다
    쉬는날시간대.columns = [f"쉬는 날×{c[2:]}시" for c in 시각열.columns]                            # 열 이름 — 「쉬는 날×9시」
    기상 = pd.DataFrame({"16도보다 추운 만큼": d["16도보다 추운 만큼"],                               # 5단계 물음 1 — 16·24도
                         "24도보다 더운 만큼": d["24도보다 더운 만큼"],
                         "3시간 동안 바뀐 만큼 (추운 쪽)": d["16도보다 추운 만큼"] - d["3시간 전 16도보다 추운 만큼"],   # 6단계 4-라 — 지금 값 − 3시간 전 값
                         "3시간 동안 바뀐 만큼 (더운 쪽)": d["24도보다 더운 만큼"] - d["3시간 전 24도보다 더운 만큼"],
                         "일사": d["icsr"]})                                                       # 5단계 물음 3 — 일사 한 열
    return pd.concat([시각열, 월열, 날열, 기상, 쉬는날시간대], axis=1)

기온두열 = ["16도보다 추운 만큼", "24도보다 더운 만큼"]                                               # 선형회귀의 지금 기온 두 열
def 시간대기온열(d):                                # 기온 × 시간대 선형회귀 — 기온 두 열 대신 시간대마다 나눈 48열, 113열(6단계 물음 6)
    나눔 = pd.DataFrame({f"{c}×{h}시": d[c] * (d["hour24"] == h) for c in 기온두열 for h in range(24)}, index=d.index)   # 그 시간대일 때만 값, 아니면 0
    return 선형열(d).drop(columns=기온두열).join(나눔)

def 트리열(d):                                     # 트리 계열에 넣는 열 — 숫자 그대로 7열(6단계 닫는 칸 ② 「입력 — 트리 계열」)
    return d[["hour24", "month", "icsr", "ta", "dow", "is_holiday", "명절"]]

입력 = {"선형회귀": 선형열(전체), "기온 × 시간대 선형회귀": 시간대기온열(전체), "트리 계열": 트리열(전체)}   # 세 해의 열을 한 번에 만든다 — 해마다 같은 열이 되게

print(f"표 모양: {len(tbl):,}행 · {tbl.shape[1] + 1}열 — 6 · 7단계가 읽은 것과 같은 파일")
쓰는곳 = {2023: "배우는 해 — 두 번 모두", 2024: "한 번은 맞히는 해, 한 번은 배우는 해", 2025: "맞히는 해 — LNG는 준비 ③에서 한 번 꺼낸다"}
display(pd.DataFrame({"시각 수": 전체.groupby("year").size(), "LNG가 있는 시각": y.notna().groupby(전체["year"]).sum(),
                      "이 노트북에서": pd.Series(쓰는곳)}).rename_axis("해"))

표 모양: 25,536행 · 25열 — 6 · 7단계가 읽은 것과 같은 파일


,시각 수,LNG가 있는 시각,이 노트북에서
해,,,
2023,8757,8757,배우는 해 — 두 번 모두
2024,8010,8010,"한 번은 맞히는 해, 한 번은 배우는 해"
2025,8760,0,맞히는 해 — LNG는 준비 ③에서 한 번 꺼낸다


In [99]:
# 8단계 모델 확정과 핵심인자 · 준비 ③
모델들 = {"선형회귀": (lambda: LinearRegression(), "선형회귀"),                                     # 7단계 2-가와 같은 넷 — 6단계 물음 6에서 정한 모델과 설정
          "기온 × 시간대 선형회귀": (lambda: LinearRegression(), "기온 × 시간대 선형회귀"),
          "랜덤포레스트": (lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=1, max_features=0.5,
                                                      n_jobs=-1, random_state=0), "트리 계열"),
          "그래디언트 부스팅": (lambda: HistGradientBoostingRegressor(early_stopping=False, random_state=0), "트리 계열")}
기준선이름 = "기준선(달·시각·날별 평균)"                                                         # 6단계 물음 2에서 가장 나았던 기준선
쉬는날 = tbl["날 종류"].isin(["토", "일", "공휴일", "명절"])                                       # 7단계와 같은 쉬는 날

원본 = pd.read_csv(os.path.join(HOT, "features_2023_2024_2025.csv"), encoding="utf-8-sig", usecols=["ts", "lng_mwh"])   # 2025년 LNG를 꺼낸다 — 이 노트북에서 여기 한 곳(7단계 2-가와 같은 줄)
원본["ts"] = pd.to_datetime(원본["ts"])

def 기준선(배운해, 시):                                                                        # 배운 해의 같은 달 · 같은 시각 · 같은 날 종류 평균으로 맞힌다(7단계 2-가 · 3-너와 같은 방법)
    yb = tbl.loc[tbl["year"].isin(배운해), "lng_mwh"]
    평균 = yb.groupby([tbl.loc[yb.index, "month"], tbl.loc[yb.index, "hour24"], 쉬는날.loc[yb.index]]).mean()
    키 = pd.MultiIndex.from_arrays([tbl.loc[시, "month"], tbl.loc[시, "hour24"], 쉬는날.loc[시]])
    return pd.Series(평균.reindex(키).values, index=시)

두번 = {}                                                                                      # 「앞 해로 배워 다음 해 맞히기」 두 번 — 맞힌 해: (실제, {방법: 예측})
for 맞힌해, 배운해 in [(2024, [2023]), (2025, [2023, 2024])]:                                   # 2023 → 2024(6단계의 연습) · 2023·2024 → 2025(7단계의 채점)
    배울곳 = 전체["year"].isin(배운해)
    맞힐곳 = 전체["year"] == 맞힌해
    시 = 전체.index[맞힐곳]                                                                     # 맞히는 해의 시각
    실제 = y[맞힐곳] if 맞힌해 == 2024 else 원본.set_index("ts")["lng_mwh"].reindex(시)          # 2024년은 준비 ②의 LNG, 2025년은 위에서 꺼낸 LNG
    예측 = {기준선이름: 기준선(배운해, 시)}
    for 이름, (만들기, 열) in 모델들.items():                                                   # 모델 넷 — 배운 해로 배우고, 맞히는 해는 입력만으로 맞힌다
        X = 입력[열]
        예측[이름] = pd.Series(만들기().fit(X[배울곳], y[배울곳]).predict(X[맞힐곳]), index=시)
    두번[f"{맞힌해}년"] = (실제, 예측)

def R2(실, 예): return 1 - ((실 - 예) ** 2).sum() / ((실 - 실.mean()) ** 2).sum()                  # 결정계수 — 6 · 7단계와 같은 식
칠단계 = {"2024년": {"선형회귀": 0.7524, "기온 × 시간대 선형회귀": 0.7679, "랜덤포레스트": 0.7701, "그래디언트 부스팅": 0.7807},   # 7단계 1-가 [표 1] 「이 노트북 결정계수」
          "2025년": {"선형회귀": 0.7007, "기온 × 시간대 선형회귀": 0.7280, "랜덤포레스트": 0.7427, "그래디언트 부스팅": 0.7553}}  # 7단계 2-가 [표 1]
for 해, (실, 예) in 두번.items():
    같다 = all(f"{R2(실, 예[m]):.4f}" == f"{v:.4f}" for m, v in 칠단계[해].items())             # 7단계에서 나온 결정계수와 소수점 넷째 자리까지 같은가
    배운 = "2023년" if 해 == "2024년" else "2023·2024년"
    print(f"{배운}으로 배워 {해} {len(실):,}시각을 맞혔다 — 모델 넷의 결정계수가 7단계와 {'같다' if 같다 else '다르다(scikit-learn 판을 확인해 주세요)'}")

2023년으로 배워 2024년 8,010시각을 맞혔다 — 모델 넷의 결정계수가 7단계와 같다
2023·2024년으로 배워 2025년 8,760시각을 맞혔다 — 모델 넷의 결정계수가 7단계와 같다


# 물음 1 — 모델 넷 가운데 어느 것으로 9단계에 가나

> **결정** — 9단계에서 2026년을 맞힐 모델은 그래디언트 부스팅이다. 1-나 ~ 1-마 결과를 보기 전에 정한 두 가지 고르는 방법이 모두 이 모델을 골랐다

## 1-나 — 달마다 나눠 보면 모델이 기준선보다 더 틀린 달이 있나 (2024 · 2025년)

In [100]:
# 8단계 모델 확정과 핵심인자 · 1-나
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 두 해를 맞힌 값이 준비 ③에서 만들어집니다"
모델이름 = list(모델들)                                                         # 선형회귀 · 기온 × 시간대 선형회귀 · 랜덤포레스트 · 그래디언트 부스팅
해이름 = {"2024년": "2024년을 맞힘 (2023년으로 배워서)", "2025년": "2025년을 맞힘 (2023 · 2024년으로 배워서)"}   # 표에 쓰는 해 이름

차이 = {}                                                                       # (해, 모델) → 달마다 (모델의 평균 절대 오차 − 기준선의 평균 절대 오차)
for 해, (실, 예) in 두번.items():
    달 = 실.index.month                                                          # 그 시각이 몇 월인가
    기준 = (실 - 예[기준선이름]).abs().groupby(달).mean()                        # 기준선이 그 달 한 시각에 평균 몇 MWh 틀렸나
    for m in 모델이름:
        차이[(해이름[해], m)] = (실 - 예[m]).abs().groupby(달).mean() - 기준       # +면 그 달 모델이 기준선보다 더 틀렸다

표1 = pd.DataFrame(차이).reindex(range(1, 13))                                   # 열두 달 — 2024년 8월은 자료가 없어 빈칸이 된다
표1.index = [f"{d}월" for d in 표1.index]
표1.index.name = "달"
print("[표 1] 달마다 — 모델의 평균 절대 오차 − 기준선의 평균 절대 오차 (MWh, 한 시각 평균)")
print("        +면 그 달 모델이 기준선보다 더 틀렸고, −면 덜 틀렸다 · 모델마다 기준선과 비교하는 표다")
display(표1.map(lambda v: "자료 없음" if pd.isna(v) else f"{v:+,.0f}"))

앞해, 뒤해 = 해이름["2024년"], 해이름["2025년"]
표2 = pd.DataFrame({"2024년 (열한 달 가운데)": {m: int((표1[(앞해, m)] > 0).sum()) for m in 모델이름},     # 2024년에 +인 달
                    "2025년 (열두 달 가운데)": {m: int((표1[(뒤해, m)] > 0).sum()) for m in 모델이름}})    # 2025년에 +인 달
표2["두 해 합 (스물세 달 가운데)"] = 표2.sum(axis=1)                                                   # 「그대로」로 셀 때 — 한 달이라도 +면 걸린다
표2["같은 달이 두 해 모두 + (열한 달 가운데)"] = [int(((표1[(앞해, m)] > 0) & (표1[(뒤해, m)] > 0)).sum()) for m in 모델이름]   # 「덜 엄격하게」로 셀 때 — 8월은 2024년이 없어 세지 않는다
표2.index.name = "모델"
print("[표 2] 모델마다 기준선보다 더 틀린 달의 수 — [표 1]에서 +인 달을 센 것")
display(표2)

[표 1] 달마다 — 모델의 평균 절대 오차 − 기준선의 평균 절대 오차 (MWh, 한 시각 평균)
        +면 그 달 모델이 기준선보다 더 틀렸고, −면 덜 틀렸다 · 모델마다 기준선과 비교하는 표다


2024년을 맞힘 (2023년으로 배워서)                                 \
                       선형회귀 기온 × 시간대 선형회귀 랜덤포레스트 그래디언트 부스팅   
달                                                            
1월                      -87          -191   -507      -618   
2월                     -555          -574   -404      -485   
3월                     -434          -466   -481      -631   
4월                     +138           +45    -70       -93   
5월                      +47           -65   -177      -100   
6월                     -137          -252   -253      -263   
7월                     +141           -80    -68      -143   
8월                    자료 없음         자료 없음  자료 없음     자료 없음   
9월                     -987          -990   -686      -876   
10월                    -328          -326   -256      -277   
11월                     +50            +3   -344      -161   
12월                     -45          -111     +6       -95   

    2025년을 맞힘 (2023 · 2024년으로 배워서)                                 
                              선형회귀 기온 × 시간대 선형회귀 랜덤포레스트 그래디언트 부스팅  
달                                                                  
1월                             +38           -56    -61      -150  
2월                            -596          -611   -635      -826  
3월                             +37           -43   -261      -369  
4월                            -209          -314   -454      -523  
5월                            -197          -326   -400      -413  
6월                             +65           -71   -140      -220  
7월                            +470          -133   -174      -218  
8월                            +211          -181   -291      -316  
9월                            -411          -366   -793      -799  
10월                           -418          -474   -254      -257  
11월                            +36           -60   -323      -260  
12월                            +72           -38   +118       -10

[표 2] 모델마다 기준선보다 더 틀린 달의 수 — [표 1]에서 +인 달을 센 것


,2024년 (열한 달 가운데),2025년 (열두 달 가운데),두 해 합 (스물세 달 가운데),같은 달이 두 해 모두 + (열한 달 가운데)
모델,,,,
선형회귀,4,7,11,2
기온 × 시간대 선형회귀,2,0,2,0
랜덤포레스트,1,1,2,1
그래디언트 부스팅,0,0,0,0


## 1-다 — 두 번(2024 · 2025년을 맞힌 것)의 결정계수와 평균 절대 오차

**표의 숫자** — [표 1] 결정계수는 1에 가까울수록 잘 맞힌 것입니다. [표 2] 평균 절대 오차는 한 시각에 평균 몇 MWh 틀렸나(+와 −를 떼고)라서 작을수록 덜 틀린 것입니다. 두 표 모두 모델끼리 비교하는 표이고, 줄(방법)은 7단계 2-가와 같은 순서입니다.

- A방식 — [표 1]의 2025년 칸에서 가장 높은 모델이 그다음 모델보다 0.002 넘게 높으면, 배운 방향(1-마)이 거꾸로가 아닌 한 그 모델입니다. 0.002 안쪽이면 나눠 본 줄(1-라)로 갑니다.
- B방식 — 1-나에서 남은 모델끼리 두 모델씩 나란히 놓고, 두 번 모두 결정계수가 높고 평균 절대 오차가 작은지 봅니다. 나눠 본 줄(1-라)에서도 절반이 넘는 줄을 덜 틀려야 「앞선다」입니다.

In [101]:
# 8단계 모델 확정과 핵심인자 · 1-다
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 두 해를 맞힌 값이 준비 ③에서 만들어집니다"
해이름 = {"2024년": "2024년을 맞힘 (2023년으로 배워서)", "2025년": "2025년을 맞힘 (2023 · 2024년으로 배워서)"}   # 1-나와 같은 해 이름
방법들 = [기준선이름] + list(모델들)                                             # 기준선과 모델 넷 — 7단계 2-가와 같은 줄 순서

결정 = pd.DataFrame({해이름[해]: {b: R2(실, 예[b]) for b in 방법들} for 해, (실, 예) in 두번.items()})   # 준비 ③과 같은 결정계수 식
결정.index.name = "방법"
print("[표 1] 결정계수 — 처음 보는 해를 맞혔을 때 · 1에 가까울수록 잘 맞혔다")
display(결정.map(lambda v: f"{v:.4f}"))

절대 = pd.DataFrame({해이름[해]: {b: (실 - 예[b]).abs().mean() for b in 방법들} for 해, (실, 예) in 두번.items()})   # 한 시각에 평균 몇 MWh 틀렸나
절대.index.name = "방법"
print("[표 2] 평균 절대 오차 — 한 시각에 평균 몇 MWh 틀렸나(+와 −를 떼고, MWh) · 작을수록 덜 틀렸다")
display(절대.map(lambda v: f"{v:,.0f}"))

[표 1] 결정계수 — 처음 보는 해를 맞혔을 때 · 1에 가까울수록 잘 맞혔다


,2024년을 맞힘 (2023년으로 배워서),2025년을 맞힘 (2023 · 2024년으로 배워서)
방법,,
기준선(달·시각·날별 평균),0.6945,0.6678
선형회귀,0.7524,0.7007
기온 × 시간대 선형회귀,0.7679,0.7280
랜덤포레스트,0.7701,0.7427
그래디언트 부스팅,0.7807,0.7553


[표 2] 평균 절대 오차 — 한 시각에 평균 몇 MWh 틀렸나(+와 −를 떼고, MWh) · 작을수록 덜 틀렸다


,2024년을 맞힘 (2023년으로 배워서),2025년을 맞힘 (2023 · 2024년으로 배워서)
방법,,
기준선(달·시각·날별 평균),"2,342","2,723"
선형회귀,"2,146","2,652"
기온 × 시간대 선형회귀,"2,072","2,503"
랜덤포레스트,"2,049","2,421"
그래디언트 부스팅,"2,004","2,364"


## 1-라 — 나눠 본 줄에서 두 모델씩, 어느 쪽이 더 자주 덜 틀리나 (2024 · 2025년)

**표의 숫자** — [표 1]의 칸은 「줄 이름의 모델이 칸 이름의 모델보다 평균 절대 오차가 작았던 줄 수 / 나눠 본 줄 수」입니다. 예를 들어 어떤 칸이 「40 / 63」이면(지어낸 숫자), 63줄 가운데 40줄에서 줄 이름의 모델이 칸 이름의 모델보다 덜 틀렸다는 뜻입니다. 모델끼리 비교하는 표입니다. [표 2]는 나눠 본 줄이 해마다 · 가지마다 몇 개인지입니다.

- B방식 — 1-나에서 남은 모델끼리, 한쪽이 1-다에서 두 번 모두 결정계수가 높고 평균 절대 오차가 작으며 이 표에서도 나눠 본 줄 수의 절반이 넘는 줄을 덜 틀리면 「앞선다」입니다. 다른 모델 모두보다 앞선 모델이 있으면 그 모델입니다.
- A방식 — 2025년 결정계수가 가장 높은 모델과 그다음 모델의 차이가 0.002 안쪽일 때만 이 표를 봅니다. 그때는 이 표에서 절반이 넘는 줄을 덜 틀린 쪽입니다.
- 줄끼리는 같은 시각을 다르게 나눈 것이라 서로 독립이 아닙니다. 그래서 줄 수로 통계 검정은 하지 않고, 절반이 넘는가만 봅니다.

In [102]:
# 8단계 모델 확정과 핵심인자 · 1-라
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 두 해를 맞힌 값이 준비 ③에서 만들어집니다"
모델이름 = list(모델들)                                                         # 모델 넷 — 1-나와 같은 순서
요일이름 = ["월", "화", "수", "목", "금", "토", "일"]

def 나눈줄(해):                                                                  # 7단계 3-다 · 3-바 · 3-카와 같게 나눈 줄마다, 모델 넷의 평균 절대 오차
    실, 예 = 두번[해]
    오차 = pd.DataFrame({m: (실 - 예[m]).abs() for m in 모델이름})                  # 시각마다 틀린 크기(+와 −를 뗀다)
    t = 오차.index
    달 = pd.Series([f"{d}월" for d in t.month], index=t)                         # 달
    시간 = pd.Series(pd.cut(t.hour, bins=[-1, 5, 9, 15, 19, 23],
                           labels=["0~5시 (밤)", "6~9시", "10~15시 (낮)", "16~19시", "20~23시"]).astype(str), index=t)   # 3-다와 같은 시간 묶음
    기온 = pd.cut(전체.loc[t, "ta"], bins=[-np.inf, -5, 0, 16, 24, 30, np.inf],
                 labels=["영하 5도 이하", "영하 5도 ~ 0도", "0 ~ 16도", "16 ~ 24도", "24 ~ 30도", "30도 넘음"]).astype(str)   # 3-바와 같은 기온 묶음
    날 = pd.Series([요일이름[d] for d in t.dayofweek], index=t)                   # 요일
    날 = 날.where(~전체.loc[t, "날 종류"].isin(["공휴일", "명절"]), 전체.loc[t, "날 종류"])   # 공휴일 · 명절은 따로 — 3-카와 같다
    줄들 = []
    for 가지, 묶음 in [("달", 달), ("시간 묶음", 시간), ("기온 묶음", 기온), ("요일과 날 종류", 날)]:
        g = 오차.groupby(묶음).mean()                                             # 그 묶음에 든 시각의 평균 절대 오차
        g.index = pd.MultiIndex.from_product([[해], [가지], g.index])
        줄들.append(g)
    return pd.concat(줄들)

줄 = pd.concat([나눈줄("2024년"), 나눈줄("2025년")])                                  # 두 해의 나눈 줄 — 줄마다 모델 넷의 평균 절대 오차

이김 = pd.DataFrame(index=모델이름, columns=모델이름, dtype=object)
for a in 모델이름:
    for b in 모델이름:
        이김.loc[a, b] = "—" if a == b else f"{int((줄[a] < 줄[b]).sum())} / {len(줄)}"   # 줄 이름의 모델이 칸 이름의 모델보다 덜 틀린 줄 수
이김.index.name = "이 모델이 ↓ · 이 모델보다 →"
print(f"[표 1] 두 모델씩 — 줄 이름의 모델이 칸 이름의 모델보다 평균 절대 오차가 작았던 줄 수 / 나눠 본 줄 {len(줄)}개")
print("        모델끼리 비교하는 표다 · 한 칸과 대각선 건너편 칸을 더하면 나눠 본 줄 수가 된다")
display(이김)

수 = 줄.groupby(level=[0, 1]).size().unstack(0).reindex(["달", "시간 묶음", "기온 묶음", "요일과 날 종류"])   # 해 · 가지마다 줄 수
수.loc["합"] = 수.sum()
수.index.name = "나눈 가지"
print("[표 2] 나눠 본 줄이 몇 개인가 — 해마다 · 가지마다")
display(수)

[표 1] 두 모델씩 — 줄 이름의 모델이 칸 이름의 모델보다 평균 절대 오차가 작았던 줄 수 / 나눠 본 줄 63개
        모델끼리 비교하는 표다 · 한 칸과 대각선 건너편 칸을 더하면 나눠 본 줄 수가 된다


,선형회귀,기온 × 시간대 선형회귀,랜덤포레스트,그래디언트 부스팅
이 모델이 ↓ · 이 모델보다 →,,,,
선형회귀,—,3 / 63,9 / 63,6 / 63
기온 × 시간대 선형회귀,60 / 63,—,20 / 63,11 / 63
랜덤포레스트,54 / 63,43 / 63,—,9 / 63
그래디언트 부스팅,57 / 63,52 / 63,54 / 63,—


[표 2] 나눠 본 줄이 몇 개인가 — 해마다 · 가지마다


,2024년,2025년
나눈 가지,,
달,11,12
시간 묶음,5,5
기온 묶음,6,6
요일과 날 종류,9,9
합,31,32


## 1-마 — 한 가지만 바꾸면 모델이 맞히는 LNG가 4단계에서 본 쪽으로 움직이나 (배운 방향)

**어떻게 보나.** 배운 두 해의 시각을 그대로 두고 한 가지만 바꿔 모델 넷으로 다시 맞힌 뒤, 맞힌 값의 평균이 얼마나 움직였는지 봅니다. 다른 것은 모두 그대로라, 그 한 가지를 바꾼 만큼만 움직입니다. 바꾸는 폭은 흔한 범위(배운 두 해 시각의 양 끝 5%씩을 뺀 범위, 1-가에서 정함) 안입니다.

| 바꾸는 것 | 어떻게 바꾸나 | 4단계에서 본 쪽 — 결과를 보기 전에 적음 |
|---|---|---|
| 기온 — 추운 쪽 | 모든 시각을 16도 → 추운 쪽 끝(아래 5%)으로. 3시간 전 기온도 같이 바꿔, 3시간 동안 바뀐 만큼은 0 | + (추울수록 LNG가 는다 — 4단계 09-13 「기온은 18~20도가 바닥인 U자」, 5단계 1-나 바닥 16도) |
| 기온 — 더운 쪽 | 모든 시각을 24도 → 더운 쪽 끝(위 5%)으로 | + (더울수록 LNG가 는다 — 같은 U자) |
| 낮의 햇빛 | 낮 10 ~ 15시 시각만, 일사를 약한 쪽 끝 → 강한 쪽 끝(낮 시각의 아래 · 위 5%)으로 | − (낮에 태양광이 많을수록 LNG가 준다 — 4단계 물음 2 결론) |
| 시간대 | 모든 시각을 3시 → 18시로. 일사는 둘 다 0 | + (수요가 느는 저녁을 LNG가 맡는다 — 4단계 물음 2 「같이 정한 것」 ①, 3시 → 18시) |
| 날 | 모든 시각을 수요일 → 일요일로(공휴일 · 명절은 아님) | − (쉬는 날 LNG가 적다 — 4단계 09-13 주말 상관 −0.33) |

**표의 숫자** — [표 1]은 흔한 범위의 양 끝 값입니다. [표 2]의 칸은 「바꾼 뒤 맞힌 LNG의 평균 − 바꾸기 전 맞힌 LNG의 평균」(MWh, 한 시각 평균)이고, 옆에 위 표의 쪽과 같으면 「같다」, 다르면 「반대」를 적었습니다. +면 바꾼 뒤에 LNG를 더 맞힌 것입니다. 모델끼리 크기를 비교하는 표가 아니라, 모델마다 쪽이 맞는지 보는 표입니다.

- 「반대」가 없는 모델은 「고르는 순서」 2를 넘습니다.
- 「반대」가 있으면 바로 빼지 않고, 왜 다른지를 셀로 먼저 봅니다. 모델이 거꾸로 배운 것이면 그 모델을 뺍니다.
- 바꾸는 폭이 흔한 범위 안이라, 배운 범위 밖에서 트리 계열이 평평해지는 것은 이 표에 들어가지 않습니다.

In [103]:
# 8단계 모델 확정과 핵심인자 · 1-마
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 모델 넷의 설정과 열이 준비 ② · ③에서 만들어집니다"
배울곳 = 전체["year"].isin([2023, 2024])                                        # 2025년을 맞힌 모델과 같은 두 해로 배운다
틀 = 전체[배울곳].copy()                                                          # 배운 두 해의 시각 — 여기서 한 가지만 바꿔 다시 맞혀 본다
다시 = {이름: 만들기().fit(입력[열][배울곳], y[배울곳]) for 이름, (만들기, 열) in 모델들.items()}   # 준비 ③의 2025년 모델과 같은 설정 · 같은 random_state

def 평균예측(d):                                                                 # d의 시각을 모델 넷으로 맞힌 값의 평균(한 시각 MWh)
    둘 = pd.concat([d, 틀], ignore_index=True)                                    # 원래 시각을 뒤에 붙여야 0/1 열이 배울 때와 같은 열로 만들어진다
    n = len(d)
    열들 = {"선형회귀": 선형열(둘), "기온 × 시간대 선형회귀": 시간대기온열(둘), "트리 계열": 트리열(둘)}
    return {이름: float(다시[이름].predict(열들[열].iloc[:n][입력[열].columns]).mean()) for 이름, (_, 열) in 모델들.items()}

def 기온을(d, g):                                                                # 지금 기온과 3시간 전 기온을 모두 g도로 — 3시간 동안 바뀐 만큼은 0
    d = d.copy()
    d["ta"] = g
    d["3시간 전 기온"] = g
    d["16도보다 추운 만큼"] = d["3시간 전 16도보다 추운 만큼"] = max(16 - g, 0)
    d["24도보다 더운 만큼"] = d["3시간 전 24도보다 더운 만큼"] = max(g - 24, 0)
    return d

def 일사를(d, g):                                                                # 일사만 g로
    d = d.copy()
    d["icsr"] = g
    return d

def 시간을(d, h):                                                                # 모든 시각을 h시로, 일사는 0으로 — 해가 없을 때의 시간대 차이만 본다
    d = d.copy()
    d["hour24"] = h
    d["icsr"] = 0.0
    return d

def 날을(d, 종류):                                                               # 모든 시각을 수요일(평일) 또는 일요일로 — 공휴일 · 명절은 아니다
    d = d.copy()
    d["날 종류"] = 종류
    d["dow"] = 2 if 종류 == "화~금" else 6                                       # 2 = 수요일, 6 = 일요일
    d["is_holiday"] = 0
    d["명절"] = 0
    return d

낮 = 틀["hour24"].between(10, 15)                                                # 낮 10 ~ 15시
기온끝 = 틀["ta"].quantile([0.05, 0.95])                                          # 흔한 범위 — 배운 두 해 시각의 양 끝 5%씩을 뺀다
일사끝 = 틀.loc[낮, "icsr"].quantile([0.05, 0.95])                                # 낮 시각의 일사도 같은 방법으로

바꿔본것 = {                                                                       # 이름: (바꾸기 전, 바꾼 뒤, 4단계에서 본 쪽)
    "기온 16도 → 추운 쪽 끝 (4단계: +)": (기온을(틀, 16), 기온을(틀, 기온끝[0.05]), +1),
    "기온 24도 → 더운 쪽 끝 (4단계: +)": (기온을(틀, 24), 기온을(틀, 기온끝[0.95]), +1),
    "낮 일사 약한 끝 → 강한 끝 (4단계: −)": (일사를(틀[낮], 일사끝[0.05]), 일사를(틀[낮], 일사끝[0.95]), -1),
    "3시 → 18시, 일사 0 (4단계: +)": (시간을(틀, 3), 시간을(틀, 18), +1),
    "수요일 → 일요일 (4단계: −)": (날을(틀, "화~금"), 날을(틀, "일"), -1),
}
움직임 = {}
for 이름, (앞d, 뒤d, 쪽) in 바꿔본것.items():
    앞, 뒤 = 평균예측(앞d), 평균예측(뒤d)                                           # 바꾸기 전 · 바꾼 뒤 맞힌 LNG의 평균
    움직임[이름] = {m: f"{뒤[m] - 앞[m]:+,.0f} · " + ("같다" if np.sign(뒤[m] - 앞[m]) == 쪽 else "반대") for m in 모델들}

print("[표 1] 흔한 범위의 양 끝 — 배운 두 해(2023 · 2024년) 시각 가운데 아래 5% · 위 5%가 시작되는 값")
display(pd.DataFrame({"아래 끝 (5%)": [f"{기온끝[0.05]:.1f}", f"{일사끝[0.05]:.2f}"], "위 끝 (95%)": [f"{기온끝[0.95]:.1f}", f"{일사끝[0.95]:.2f}"]},
                     index=["기온 (도)", "낮 10 ~ 15시 일사 (MJ/m²)"]).rename_axis("무엇"))
print("[표 2] 한 가지만 바꿨을 때 모델이 맞힌 LNG가 움직인 양 (MWh, 한 시각 평균) — 바꾼 뒤 − 바꾸기 전")
print("        +면 바꾼 뒤에 LNG를 더 맞혔다 · 「같다」면 칸 이름의 괄호(4단계에서 본 쪽)와 같은 쪽이다 · 모델마다 쪽을 보는 표다")
display(pd.DataFrame(움직임).rename_axis("모델"))

[표 1] 흔한 범위의 양 끝 — 배운 두 해(2023 · 2024년) 시각 가운데 아래 5% · 위 5%가 시작되는 값


,아래 끝 (5%),위 끝 (95%)
무엇,,
기온 (도),-3.2,29.1
낮 10 ~ 15시 일사 (MJ/m²),0.21,3.28


[표 2] 한 가지만 바꿨을 때 모델이 맞힌 LNG가 움직인 양 (MWh, 한 시각 평균) — 바꾼 뒤 − 바꾸기 전
        +면 바꾼 뒤에 LNG를 더 맞혔다 · 「같다」면 칸 이름의 괄호(4단계에서 본 쪽)와 같은 쪽이다 · 모델마다 쪽을 보는 표다


,기온 16도 → 추운 쪽 끝 (4단계: +),기온 24도 → 더운 쪽 끝 (4단계: +),낮 일사 약한 끝 → 강한 끝 (4단계: −),"3시 → 18시, 일사 0 (4단계: +)",수요일 → 일요일 (4단계: −)
모델,,,,,
선형회귀,"+5,401 · 같다","+4,985 · 같다","-7,995 · 같다","+10,534 · 같다","-5,235 · 같다"
기온 × 시간대 선형회귀,"+5,046 · 같다","+2,994 · 같다","-7,476 · 같다","+10,894 · 같다","-5,249 · 같다"
랜덤포레스트,"+3,776 · 같다","+2,779 · 같다","-6,229 · 같다","+9,008 · 같다","-5,098 · 같다"
그래디언트 부스팅,"+4,112 · 같다","+2,722 · 같다","-6,903 · 같다","+9,163 · 같다","-5,201 · 같다"


# 물음 2 — 어느 변수가 LNG를 얼마나 움직였나 (핵심인자)

> **결정** — 모델에 넣은 다섯 모두 핵심인자이고, 몇 시인가 → 무슨 날인가 → 기온 → 햇빛 → 몇 월인가 순서로 LNG를 움직였다

## 2-나 — 변수마다 LNG를 얼마나 움직였나 (흔히 있는 값을 다 넣어 봤을 때의 폭, 모델 넷)

**표의 숫자** — [표 1]의 칸은 변수 하나만 바꿔 가며 맞힌 LNG(한 시각 평균)의 가장 큰 값 − 가장 작은 값입니다(MWh). 늘 0 이상이고, 클수록 그 변수가 LNG를 크게 움직였다는 뜻입니다. 햇빛 줄만 낮 10 ~ 15시 시각의 평균입니다. 괄호 안은 그 모델 안에서의 순서입니다. 변수끼리 크기를 비교하는 표이고, 모델끼리 비교하는 표가 아닙니다. [표 2]는 그래디언트 부스팅에서 그 폭이 어디서 나왔는지, 곧 가장 많이 맞힌 값과 가장 적게 맞힌 값을 보입니다.

**넣어 보는 값**

| 변수 | 넣어 보는 값 | 같이 정해 둔 것 |
|---|---|---|
| 기온 | 흔한 범위(배운 두 해 시각의 양 끝 5%씩을 뺀 범위)의 양 끝과 그 사이 1도마다 | 3시간 전 기온도 같은 값으로 넣어, 3시간 동안 바뀐 만큼은 0(1-마와 같다) |
| 햇빛(일사) | 낮 10 ~ 15시 시각만, 흔한 범위를 열여섯 값으로 | 밤에는 늘 0이라 바꿀 수 없어서 낮만 본다 |
| 몇 시인가 | 0 ~ 23시 | 일사는 0으로 — 밤에 해가 뜬 시각이 생기지 않게(1-마와 같다) |
| 몇 월인가 | 1 ~ 12월 | — |
| 무슨 날인가 | 월 · 화~금 · 토 · 일 · 공휴일 · 명절 | 트리 계열에는 화~금 · 공휴일 · 명절을 수요일로 넣는다 |

- 2-가에서 정한 대로, [표 1] 「넷의 평균」이 큰 순서가 LNG를 크게 움직인 순서입니다.
- 괄호 안 순서가 모델마다 크게 다른 변수는 따로 적습니다.
- 명절처럼 일 년에 며칠 안 되는 날이 위에 들 수 있습니다. 일 년 전체로 볼 때 그 변수를 뒤섞으면 얼마나 더 틀리는지는 다음 셀(순열 중요도)에서 봅니다.
- 달과 기온은 같이 움직이는 변수라서, 하나만 바꾸면 7월인데 영하인 시각처럼 실제로는 없는 시각이 생깁니다. 트리 계열은 그런 시각을 배운 적이 없어서, 달과 기온의 폭은 조금 다르게 나올 수 있습니다.

In [104]:
# 8단계 모델 확정과 핵심인자 · 2-나
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 모델 넷의 설정과 열이 준비 ② · ③에서 만들어집니다"
배울곳 = 전체["year"].isin([2023, 2024])                                        # 1-마와 같은 두 해로 배운다 — 2025년 LNG는 쓰지 않는다
틀 = 전체[배울곳].copy()                                                          # 배운 두 해의 시각 — 여기서 한 가지만 바꿔 다시 맞혀 본다
다시 = {이름: 만들기().fit(입력[열][배울곳], y[배울곳]) for 이름, (만들기, 열) in 모델들.items()}   # 1-마와 같은 설정 · 같은 random_state라 1-마와 같은 모델

def 평균예측(d):                                                                 # d의 시각을 모델 넷으로 맞힌 값의 평균(한 시각 MWh) — 1-마와 같다
    둘 = pd.concat([d, 틀], ignore_index=True)                                    # 원래 시각을 뒤에 붙여야 0/1 열이 배울 때와 같은 열로 만들어진다
    n = len(d)
    열들 = {"선형회귀": 선형열(둘), "기온 × 시간대 선형회귀": 시간대기온열(둘), "트리 계열": 트리열(둘)}
    return {이름: float(다시[이름].predict(열들[열].iloc[:n][입력[열].columns]).mean()) for 이름, (_, 열) in 모델들.items()}

def 기온을(d, g):                                                                # 지금 기온과 3시간 전 기온을 모두 g도로 — 3시간 동안 바뀐 만큼은 0(1-마와 같다)
    d = d.copy()
    d["ta"] = g
    d["3시간 전 기온"] = g
    d["16도보다 추운 만큼"] = d["3시간 전 16도보다 추운 만큼"] = max(16 - g, 0)
    d["24도보다 더운 만큼"] = d["3시간 전 24도보다 더운 만큼"] = max(g - 24, 0)
    return d

def 일사를(d, g):                                                                # 일사만 g로
    d = d.copy()
    d["icsr"] = g
    return d

def 시간을(d, h):                                                                # 모든 시각을 h시로, 일사는 0으로 — 밤에 해가 뜬 시각이 생기지 않게(1-마와 같다)
    d = d.copy()
    d["hour24"] = h
    d["icsr"] = 0.0
    return d

def 달을(d, m):                                                                  # 모든 시각을 m월로
    d = d.copy()
    d["month"] = m
    return d

날값 = {"월": (0, 0, 0), "화~금": (2, 0, 0), "토": (5, 0, 0), "일": (6, 0, 0), "공휴일": (2, 1, 0), "명절": (2, 1, 1)}   # (요일, 공휴일, 명절) — 화~금 · 공휴일 · 명절은 수요일로 넣는다
def 날을(d, 종류):                                                               # 모든 시각을 그 날 종류로
    d = d.copy()
    d["날 종류"] = 종류
    d["dow"], d["is_holiday"], d["명절"] = 날값[종류]
    return d

낮 = 틀["hour24"].between(10, 15)                                                # 낮 10 ~ 15시
기온끝 = 틀["ta"].quantile([0.05, 0.95])                                          # 흔한 범위 — 배운 두 해 시각의 양 끝 5%씩을 뺀다(1-마와 같다)
일사끝 = 틀.loc[낮, "icsr"].quantile([0.05, 0.95])                                # 낮 시각의 일사도 같은 방법으로
훑기 = {                                                                          # 변수: (어느 시각에서, 넣어 볼 값들, 바꾸는 함수, 값을 적는 방법)
    "기온": (틀, np.unique(np.r_[기온끝[0.05], np.arange(np.ceil(기온끝[0.05]), np.floor(기온끝[0.95]) + 1), 기온끝[0.95]]), 기온을, lambda v: f"{v:g}도"),   # 양 끝(1-마와 같은 값)과 그 사이 1도마다
    "햇빛(일사) — 낮 10 ~ 15시만": (틀[낮], np.linspace(일사끝[0.05], 일사끝[0.95], 16), 일사를, lambda v: f"{v:.2f} MJ/m²"),
    "몇 시인가 — 일사 0": (틀, range(24), 시간을, lambda v: f"{v}시"),
    "몇 월인가": (틀, range(1, 13), 달을, lambda v: f"{v}월"),
    "무슨 날인가": (틀, list(날값), 날을, lambda v: v),
}
폭, 끝 = {}, {}
for 변수, (d, 값들, 바꾸기, 적기) in 훑기.items():
    맞힘 = pd.DataFrame({v: 평균예측(바꾸기(d, v)) for v in 값들})               # 줄: 모델, 칸: 넣어 본 값 — 칸마다 맞힌 LNG의 평균
    폭[변수] = 맞힘.max(axis=1) - 맞힘.min(axis=1)                                # 가장 많이 맞힌 값 − 가장 적게 맞힌 값
    g = 맞힘.loc["그래디언트 부스팅"]                                              # [표 2]에 계산을 보일 모델
    끝[변수] = {"가장 많이 맞힌 때": f"{적기(g.idxmax())} · {g.max():,.0f}",
                "가장 적게 맞힌 때": f"{적기(g.idxmin())} · {g.min():,.0f}",
                "폭 = 앞 − 뒤": f"{g.max() - g.min():,.0f}"}

표 = pd.DataFrame(폭).T                                                           # 줄: 변수, 칸: 모델
평균 = 표.mean(axis=1)
순위 = 표.rank(ascending=False).astype(int)                                       # 모델 안에서의 순서
보기 = 표.apply(lambda c: [f"{v:,.0f} ({r})" for v, r in zip(c, 순위[c.name])])   # 「폭 (그 모델 안의 순서)」
보기["넷의 평균"] = [f"{v:,.0f}" for v in 평균]
보기 = 보기.loc[평균.sort_values(ascending=False).index]
보기.insert(0, "순서", range(1, len(보기) + 1))

print("[표 1] 변수마다 LNG를 움직인 폭 (MWh, 한 시각 평균) — 흔히 있는 값을 다 넣어 봤을 때 가장 많이 맞힌 값 − 가장 적게 맞힌 값")
print("        괄호는 그 모델 안의 순서 · 「넷의 평균」이 큰 순서로 줄을 놓았다 · 햇빛 줄만 낮 10 ~ 15시 시각의 평균이다")
display(보기.rename_axis("변수"))
print("[표 2] 그래디언트 부스팅에서 [표 1]의 폭이 나온 계산 — 어디서 가장 많이 · 가장 적게 맞혔나 (MWh, 한 시각 평균)")
display(pd.DataFrame(끝).T.loc[보기.index].rename_axis("변수"))

[표 1] 변수마다 LNG를 움직인 폭 (MWh, 한 시각 평균) — 흔히 있는 값을 다 넣어 봤을 때 가장 많이 맞힌 값 − 가장 적게 맞힌 값
        괄호는 그 모델 안의 순서 · 「넷의 평균」이 큰 순서로 줄을 놓았다 · 햇빛 줄만 낮 10 ~ 15시 시각의 평균이다


,순서,선형회귀,기온 × 시간대 선형회귀,랜덤포레스트,그래디언트 부스팅,넷의 평균
변수,,,,,,
몇 시인가 — 일사 0,1,"12,103 (1)","12,688 (1)","9,008 (1)","9,547 (1)","10,837"
햇빛(일사) — 낮 10 ~ 15시만,2,"7,995 (2)","7,476 (2)","6,229 (2)","6,903 (2)","7,150"
무슨 날인가,3,"6,257 (3)","6,154 (3)","5,098 (3)","5,201 (3)","5,677"
기온,4,"5,401 (4)","5,046 (4)","3,832 (4)","4,125 (4)","4,601"
몇 월인가,5,"2,680 (5)","2,636 (5)","1,630 (5)","2,104 (5)","2,262"


[표 2] 그래디언트 부스팅에서 [표 1]의 폭이 나온 계산 — 어디서 가장 많이 · 가장 적게 맞혔나 (MWh, 한 시각 평균)


,가장 많이 맞힌 때,가장 적게 맞힌 때,폭 = 앞 − 뒤
변수,,,
몇 시인가 — 일사 0,"16시 · 22,134","3시 · 12,587","9,547"
햇빛(일사) — 낮 10 ~ 15시만,"0.21 MJ/m² · 22,481","3.28 MJ/m² · 15,578","6,903"
무슨 날인가,"화~금 · 19,464","일 · 14,263","5,201"
기온,"-3.2도 · 19,656","19도 · 15,531","4,125"
몇 월인가,"3월 · 19,007","11월 · 16,903","2,104"


## 2-다 — 변수 하나의 값만 뒤섞으면 일 년 전체에서 얼마나 더 틀리나 (순열 중요도, 모델 넷)

**어떻게 내나.** 2023년으로 배운 모델 넷으로 2024년을 맞히면서, 변수 하나의 값만 시각끼리 뒤섞습니다. 예를 들어 기온을 뒤섞으면 7월 오후 시각에 1월 새벽의 기온이 들어갑니다. 모델이 그 변수를 많이 썼다면 그만큼 많이 틀리게 됩니다. 뒤섞은 뒤 평균 절대 오차 − 원래 평균 절대 오차가 그 변수의 크기이고, 다섯 번 뒤섞어 평균 냅니다. 한 변수에서 나온 열(기온이면 16도보다 추운 만큼 · 3시간 전 기온 같은 것)은 같이 뒤섞습니다. 6단계에서 연습한 것과 같은 「2023년으로 배워 2024년 맞히기」라서 2025년 LNG는 쓰지 않습니다. 돌리는 데 10초쯤 걸립니다.

**표의 숫자** — [표 1]의 칸은 뒤섞어서 늘어난 평균 절대 오차입니다(MWh, 한 시각 평균). 클수록 그 모델이 그 변수를 많이 썼다는 뜻이고, 0이면 그 변수가 없어도 같다는 뜻입니다. 괄호는 그 모델 안에서 몇 번째인지입니다. 변수끼리 비교하는 표입니다. [표 2]는 그래디언트 부스팅에서 이 숫자가 나온 계산, 곧 원래 평균 절대 오차와 뒤섞은 뒤 평균 절대 오차를 보입니다. [표 3]은 이 셀의 물음에 바로 답하는 표입니다. 변수마다 2-나(LNG를 움직인 폭)와 2-다(뒤섞었을 때 늘어난 오차)에서 몇 번째인지를 나란히 놓았습니다.

- 「순서」가 「2-나의 순서」와 같으면, 핵심인자의 순서가 방법에 따라 달라지지 않는다는 뜻입니다.
- 다르면 어느 변수가 다른지 봅니다. 햇빛은 2-나에서 낮 시각만 봤으니 여기서는 작게 나올 수 있습니다. 무슨 날인가는 명절 · 공휴일 같은 드문 날이 2-나의 폭을 키웠다면 여기서 작게 나올 수 있습니다.
- 뒤섞기도 2-나처럼 실제로는 없는 시각(7월 새벽에 영하)을 만듭니다. 그래서 달과 기온처럼 같이 움직이는 변수는 숫자가 조금 다르게 나올 수 있습니다.

In [105]:
# 8단계 모델 확정과 핵심인자 · 2-다
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 모델 넷의 설정과 열이 준비 ② · ③에서 만들어집니다"
배운곳 = 전체["year"] == 2023                                                    # 6단계의 연습과 같다 — 2023년으로 배워
맞힌곳 = 전체["year"] == 2024                                                    # 2024년을 맞힌다(2025년 LNG는 쓰지 않는다)
연습 = {이름: 만들기().fit(입력[열][배운곳], y[배운곳]) for 이름, (만들기, 열) in 모델들.items()}   # 준비 ③의 2024년 모델과 같은 설정 · 같은 random_state
틀24 = 전체[맞힌곳].copy()                                                         # 2024년 시각 — 여기서 변수 하나의 값만 시각끼리 뒤섞는다
실24 = y[맞힌곳]                                                                   # 2024년 실제 LNG

def 오차(d):                                                                     # d의 시각을 모델 넷으로 맞혔을 때 평균 절대 오차(한 시각 MWh)
    둘 = pd.concat([d, 전체[배운곳]], ignore_index=True)                           # 2023년 시각을 뒤에 붙여야 0/1 열이 배울 때와 같은 열로 만들어진다(2024년에는 8월이 없다)
    n = len(d)
    열들 = {"선형회귀": 선형열(둘), "기온 × 시간대 선형회귀": 시간대기온열(둘), "트리 계열": 트리열(둘)}
    return {이름: float(np.abs(실24.values - 연습[이름].predict(열들[열].iloc[:n][입력[열].columns])).mean()) for 이름, (_, 열) in 모델들.items()}

묶음 = {                                                                          # 변수 하나 = 같이 뒤섞을 열들(한 변수에서 나온 열은 같은 순서로 뒤섞어야 서로 맞는다)
    "몇 시인가": ["hour24"],
    "햇빛(일사)": ["icsr"],
    "무슨 날인가": ["날 종류", "dow", "is_holiday", "명절"],
    "기온": ["ta", "3시간 전 기온", "16도보다 추운 만큼", "24도보다 더운 만큼", "3시간 전 16도보다 추운 만큼", "3시간 전 24도보다 더운 만큼"],
    "몇 월인가": ["month"],
}
나순서 = {"몇 시인가": 1, "햇빛(일사)": 2, "무슨 날인가": 3, "기온": 4, "몇 월인가": 5}   # 2-나 [표 1]의 순서(넷의 평균)

원래 = 오차(틀24)                                                                  # 뒤섞기 전 평균 절대 오차
늘어남, 섞은뒤 = {}, {}
for 변수, 열 in 묶음.items():
    합 = {m: 0.0 for m in 모델들}
    for 번 in range(5):                                                           # 다섯 번 뒤섞어 평균 낸다 — 뒤섞는 순서에 따라 조금씩 달라서
        d = 틀24.copy()
        섞음 = np.random.default_rng(번).permutation(len(d))                       # 시각의 순서를 무작위로 바꾼 번호
        d[열] = 틀24[열].values[섞음]                                                # 그 변수의 값만 다른 시각의 값으로 바꾼다
        for m, v in 오차(d).items(): 합[m] += v / 5
    섞은뒤[변수] = 합
    늘어남[변수] = {m: 합[m] - 원래[m] for m in 모델들}                               # 뒤섞은 뒤 오차 − 원래 오차

표 = pd.DataFrame(늘어남).T                                                       # 줄: 변수, 칸: 모델
평균 = 표.mean(axis=1)
순위 = 표.rank(ascending=False).astype(int)
보기 = 표.apply(lambda c: [f"{v:,.0f} ({r})" for v, r in zip(c, 순위[c.name])])   # 「늘어난 양 (그 모델 안에서 몇 번째)」
보기["넷의 평균"] = [f"{v:,.0f}" for v in 평균]
보기 = 보기.loc[평균.sort_values(ascending=False).index]
다순서 = {v: i + 1 for i, v in enumerate(보기.index)}                              # 넷의 평균으로 몇 번째인가

print("[표 1] 변수 하나의 값만 시각끼리 뒤섞었을 때 평균 절대 오차가 늘어난 양 (MWh, 한 시각 평균) — 2023년으로 배워 2024년을 맞힘")
print("        괄호는 그 모델 안에서 몇 번째인가 · 「넷의 평균」이 큰 순서로 줄을 놓았다 · 다섯 번 뒤섞어 평균 냈다")
display(보기.rename_axis("변수"))
g = "그래디언트 부스팅"
print(f"[표 2] {g}에서 [표 1]이 나온 계산 — 원래 평균 절대 오차 {원래[g]:,.0f} MWh")
display(pd.DataFrame({"뒤섞은 뒤 평균 절대 오차": [f"{섞은뒤[v][g]:,.0f}" for v in 보기.index],
                      "늘어난 양 = 앞 − 원래": [f"{섞은뒤[v][g] - 원래[g]:,.0f}" for v in 보기.index]}, index=보기.index).rename_axis("변수"))
print("[표 3] 2-나와 2-다에서 몇 번째인가 — 둘 다 모델 넷의 평균으로 · 줄은 2-나의 순서대로")
비교 = pd.DataFrame({"2-나에서 몇 번째 (LNG를 움직인 폭)": 나순서, "2-다에서 몇 번째 (뒤섞었을 때 늘어난 오차)": 다순서})
비교["2-나에서 2-다로"] = ["같다" if a == b else ("올라감" if b < a else "내려감") for a, b in zip(비교.iloc[:, 0], 비교.iloc[:, 1])]
display(비교.sort_values(비교.columns[0]).rename_axis("변수"))

[표 1] 변수 하나의 값만 시각끼리 뒤섞었을 때 평균 절대 오차가 늘어난 양 (MWh, 한 시각 평균) — 2023년으로 배워 2024년을 맞힘
        괄호는 그 모델 안에서 몇 번째인가 · 「넷의 평균」이 큰 순서로 줄을 놓았다 · 다섯 번 뒤섞어 평균 냈다


,선형회귀,기온 × 시간대 선형회귀,랜덤포레스트,그래디언트 부스팅,넷의 평균
변수,,,,,
몇 시인가,"3,343 (1)","3,589 (1)","2,114 (1)","2,648 (1)","2,923"
무슨 날인가,"1,422 (2)","1,430 (2)","1,390 (2)","1,431 (2)","1,418"
기온,"1,103 (4)","1,180 (4)",786 (3),940 (3),"1,002"
햇빛(일사),"1,329 (3)","1,218 (3)",453 (4),582 (4),896
몇 월인가,214 (5),225 (5),359 (5),323 (5),280


[표 2] 그래디언트 부스팅에서 [표 1]이 나온 계산 — 원래 평균 절대 오차 2,004 MWh


,뒤섞은 뒤 평균 절대 오차,늘어난 양 = 앞 − 원래
변수,,
몇 시인가,"4,651","2,648"
무슨 날인가,"3,434","1,431"
기온,"2,944",940
햇빛(일사),"2,585",582
몇 월인가,"2,326",323


[표 3] 2-나와 2-다에서 몇 번째인가 — 둘 다 모델 넷의 평균으로 · 줄은 2-나의 순서대로


,2-나에서 몇 번째 (LNG를 움직인 폭),2-다에서 몇 번째 (뒤섞었을 때 늘어난 오차),2-나에서 2-다로
변수,,,
몇 시인가,1,1,같다
햇빛(일사),2,4,내려감
무슨 날인가,3,2,올라감
기온,4,3,올라감
몇 월인가,5,5,같다


## 2-라 — 실제 LNG를 변수의 값마다 묶어 평균 내면, 변수마다 LNG가 얼마나 달랐나 (모델 없이 · 탐색적 기법 결과)

**어떻게 내나.** 모델이 맞힌 값은 쓰지 않습니다. 「시각」은 한 시간짜리 기록 하나입니다(2023년 1월 1일 18시가 시각 하나). 두 해의 시각을 그 변수의 값이 같은 것끼리 묶습니다. 몇 시인가라면 0시인 시각끼리, 1시인 시각끼리, … 23시인 시각끼리 스물네 묶음입니다. 묶음마다 실제 LNG를 평균 내고, 그 평균 가운데 가장 많은 것에서 가장 적은 것을 뺀 값이 그 변수의 폭입니다. 이렇게 묶으면 묶음끼리 날씨 같은 다른 조건을 같게 맞출 수 없습니다. 그래서 예를 들어 1월 묶음의 평균에는 추운 날씨 때문에 늘어난 LNG가 섞여 있습니다.

**표의 숫자**

- 모두 한 시각의 LNG 발전량(MWh)입니다. 폭은 그런 평균 둘을 뺀 값입니다.
- 햇빛 줄만 낮 10 ~ 15시(10 · 11 · 12 · 13 · 14 · 15시, 여섯 시간)의 시각으로 냅니다. 밤에는 햇빛이 늘 0이라서 2-나와 같게 낮만 봅니다.
- 기온과 햇빛은 흔한 범위 안의 시각만 씁니다. 흔한 범위는 값을 작은 것부터 줄 세웠을 때 양 끝 5%씩을 뺀 범위입니다(2-나와 같습니다. 햇빛은 낮 시각으로 정합니다). 범위의 두 끝은 [표 2] 「어떻게 묶었나」 칸에 있습니다.
- [표 1] — 두 폭과, 그 폭으로 다섯 변수를 세운 순서입니다. 「모델이 맞힌 값으로 본 폭」은 2-나 [표 1]의 「넷의 평균」 칸을 옮겨 적은 것입니다. 9단계에 가는 모델은 그래디언트 부스팅 하나지만, 핵심인자는 모델 하나로 정하지 않기로 해서(2-가) 순서를 모델 넷의 평균으로 세웁니다.
- [표 2] — [표 1]의 「실제 LNG로 본 폭」이 나온 계산입니다. 변수마다 어떻게 묶었는지, 평균이 가장 많은 묶음과 가장 적은 묶음, 그 뺄셈을 보입니다.
- [표 3] — 그 계산을 몇 월인가 하나로 열두 묶음 모두 보인 것입니다. 옆에 달만 그 달로 바꿨을 때 모델이 맞힌 평균을 놓았습니다. 이 칸은 계산을 보이려고 모델 넷 가운데 그래디언트 부스팅(물음 1에서 고른 모델) 하나로 냈습니다. 그래서 [표 1]의 모델 칸(모델 넷의 평균)과 숫자가 다릅니다.

- 모델 없이 본 폭이 2-나보다 크면, 그 변수의 값마다 다른 변수도 같이 달라졌다는 뜻입니다(예: 1월에는 날씨도 춥다).
- 순서가 2-나와 다르면, 다른 변수를 그대로 두고 하나씩 봐야 순서가 정해진다는 뜻입니다. 그것이 이 프로젝트에서 모델을 쓰는 까닭입니다.

In [106]:
# 8단계 모델 확정과 핵심인자 · 2-라
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 준비 ② · ③에서 표와 열이 만들어집니다"
배운곳 = 전체["year"].isin([2023, 2024])                                            # 배운 두 해의 시각 — 2025년 LNG는 쓰지 않는다
d = 전체[배운곳].copy()
d["LNG"] = y[d.index]                                                             # 그 시각에 실제로 기록된 LNG(MWh) — 모델이 맞힌 값이 아니다
기온끝 = d["ta"].quantile([0.05, 0.95])                                            # 흔한 범위 — 작은 것부터 줄 세워 양 끝 5%씩을 뺀다(2-나와 같다)
낮 = d["hour24"].between(10, 15)                                                  # 낮 10 ~ 15시(10 · 11 · 12 · 13 · 14 · 15시)
일사끝 = d.loc[낮, "icsr"].quantile([0.05, 0.95])                                  # 햇빛의 흔한 범위는 낮 시각으로 정한다
온d = d[d["ta"].between(기온끝[0.05], 기온끝[0.95])]                               # 흔한 범위 안의 기온인 시각
낮d = d[낮 & d["icsr"].between(일사끝[0.05], 일사끝[0.95])]                         # 흔한 범위 안의 햇빛인 낮 시각

def 묶어서(시각들, 값):                                                             # 값이 같은 시각끼리 묶어, 묶음마다 시각 수와 실제 LNG의 평균을 낸다
    g = 시각들.groupby(값, observed=True)["LNG"]
    return pd.DataFrame({"시각 수": g.size(), "평균": g.mean().round()})          # 평균을 반올림해 두어야 아래 뺄셈이 표에 적힌 숫자로 맞는다

묶음들 = {                                                                         # 변수: (어떻게 묶었나, 묶음마다 시각 수와 평균)
    "몇 시인가": ("0 ~ 23시마다", 묶어서(d, d["hour24"]).rename(index=lambda v: f"{v}시")),
    "햇빛(일사) — 낮 10 ~ 15시만": (f"낮 시각의 흔한 범위({일사끝[0.05]:.2f} ~ {일사끝[0.95]:.2f} MJ/m²)를 같은 너비의 열다섯 구간으로",
                                   묶어서(낮d, pd.cut(낮d["icsr"], np.linspace(일사끝[0.05], 일사끝[0.95], 16), include_lowest=True))
                                   .rename(index=lambda v: f"{v.left:.2f} ~ {v.right:.2f} MJ/m²")),
    "무슨 날인가": ("날 종류(월 · 화~금 · 토 · 일 · 공휴일 · 명절)마다", 묶어서(d, d["날 종류"])),
    "기온": (f"흔한 범위({기온끝[0.05]:.1f} ~ {기온끝[0.95]:.1f}도) 안에서 반올림해 1도마다", 묶어서(온d, 온d["ta"].round()).rename(index=lambda v: f"{v:.0f}도")),
    "몇 월인가": ("1 ~ 12월마다", 묶어서(d, d["month"]).rename(index=lambda v: f"{v}월")),
}
많은 = {v: m["평균"].idxmax() for v, (_, m) in 묶음들.items()}                      # 실제 LNG 평균이 가장 많은 묶음
적은 = {v: m["평균"].idxmin() for v, (_, m) in 묶음들.items()}                      # 실제 LNG 평균이 가장 적은 묶음
폭 = pd.Series({v: m.loc[많은[v], "평균"] - m.loc[적은[v], "평균"] for v, (_, m) in 묶음들.items()})
나폭 = pd.Series({"몇 시인가": 10837, "햇빛(일사) — 낮 10 ~ 15시만": 7150, "무슨 날인가": 5677, "기온": 4601, "몇 월인가": 2262})   # 2-나 [표 1] 「넷의 평균」(10-12에 돌린 값)
줄 = 나폭.sort_values(ascending=False).index                                       # 줄은 2-나의 순서대로

print("[표 1] 변수마다 LNG가 얼마나 달랐나 (한 시각의 LNG, MWh) — 실제 LNG로 본 폭(모델 없이)과 모델이 맞힌 값으로 본 폭(2-나)")
display(pd.DataFrame({"실제 LNG로 본 폭 (모델 없이)": 폭.map("{:,.0f}".format),
                      "모델이 맞힌 값으로 본 폭 (2-나 · 모델 넷이 각각 낸 폭의 평균)": 나폭.map("{:,.0f}".format),
                      "실제 LNG로 몇 번째 (1이 가장 큼)": 폭.rank(ascending=False).astype(int),
                      "모델로 몇 번째 (2-나 · 1이 가장 큼)": 나폭.rank(ascending=False).astype(int)}).loc[줄].rename_axis("변수"))

print("[표 2] [표 1]의 「실제 LNG로 본 폭」이 나온 계산 — 두 해의 시각을 그 변수의 값이 같은 것끼리 묶고, 묶음마다 실제 LNG를 평균 내서, 평균이 가장 많은 묶음 − 가장 적은 묶음")
display(pd.DataFrame({"어떻게 묶었나": {v: f"{방법} — {len(m)}묶음" for v, (방법, m) in 묶음들.items()},
                      "실제 LNG 평균이 가장 많은 묶음 (그 묶음의 시각 수)": {v: f"{많은[v]} (시각 {m.loc[많은[v], '시각 수']:,}개)" for v, (_, m) in 묶음들.items()},
                      "실제 LNG 평균이 가장 적은 묶음 (그 묶음의 시각 수)": {v: f"{적은[v]} (시각 {m.loc[적은[v], '시각 수']:,}개)" for v, (_, m) in 묶음들.items()},
                      "폭 = 가장 많은 묶음의 평균 − 가장 적은 묶음의 평균": {v: f"{m.loc[많은[v], '평균']:,.0f} − {m.loc[적은[v], '평균']:,.0f} = {폭[v]:,.0f}"
                                                                 for v, (_, m) in 묶음들.items()}}).loc[줄].rename_axis("변수"))

gb = 모델들["그래디언트 부스팅"][0]().fit(입력["트리 계열"][배운곳], y[배운곳])      # 2-나와 같은 두 해 · 같은 설정으로 배운다 — 2-나의 그래디언트 부스팅과 같은 모델
x = 입력["트리 계열"][배운곳].copy()                                              # 두 해의 모든 시각
gb월 = {}
for 달 in range(1, 13):
    x["month"] = 달                                                               # 모든 시각을 그 달로 — 기온 · 햇빛 · 몇 시 · 무슨 날은 시각마다 그대로
    gb월[f"{달}월"] = round(float(gb.predict(x).mean()))                          # 모델이 맞힌 LNG의 평균
gb월 = pd.Series(gb월)
월 = 묶음들["몇 월인가"][1]
print("[표 3] [표 2]의 계산을 몇 월인가로 열두 묶음 모두 — 달마다 LNG 평균 (한 시각의 LNG, MWh · 2023 · 2024년)")
print("        「모델이 맞힌 평균」은 두 해의 모든 시각에서 달만 그 달로 바꿔 넣고 그래디언트 부스팅(물음 1에서 고른 모델)이 맞힌 LNG의 평균이다.")
print("        예를 들어 7월 14시 시각이면 기온 · 햇빛 · 몇 시 · 무슨 날은 그 시각의 값 그대로 두고 달만 1월로 넣는다.")
print("        열두 달 모두 같은 날씨 · 같은 시각으로 맞히니, 달끼리의 차이는 달 하나만 바꾼 차이다.")
display(pd.DataFrame({"그 달의 시각 수": 월["시각 수"].map("{:,}".format),
                      "그 달의 평균 기온 (도)": d.groupby("month")["ta"].mean().rename(index=lambda v: f"{v}월").map("{:.1f}".format),
                      "실제 LNG의 평균 (모델 없이)": 월["평균"].map("{:,.0f}".format),
                      "모델이 맞힌 평균 (그래디언트 부스팅 · 모든 시각을 그 달로)": gb월.map("{:,}".format)}).rename_axis("달"))
print(f"실제 LNG — 가장 많은 {많은['몇 월인가']} {월['평균'].max():,.0f} − 가장 적은 {적은['몇 월인가']} {월['평균'].min():,.0f} = {폭['몇 월인가']:,.0f}"
      f" ([표 1]의 몇 월인가 「실제 LNG로 본 폭」과 같다)")
print(f"모델이 맞힌 값 — 가장 많은 {gb월.idxmax()} {gb월.max():,} − 가장 적은 {gb월.idxmin()} {gb월.min():,} = {gb월.max() - gb월.min():,}"
      f" (2-나에서 그래디언트 부스팅이 낸 몇 월인가 폭과 같다. [표 1]의 {나폭['몇 월인가']:,}은 모델 넷이 낸 폭의 평균이라 다르다)")

[표 1] 변수마다 LNG가 얼마나 달랐나 (한 시각의 LNG, MWh) — 실제 LNG로 본 폭(모델 없이)과 모델이 맞힌 값으로 본 폭(2-나)


,실제 LNG로 본 폭 (모델 없이),모델이 맞힌 값으로 본 폭 (2-나 · 모델 넷이 각각 낸 폭의 평균),실제 LNG로 몇 번째 (1이 가장 큼),모델로 몇 번째 (2-나 · 1이 가장 큼)
변수,,,,
몇 시인가,"9,782","10,837",1,1
햇빛(일사) — 낮 10 ~ 15시만,"6,034","7,150",3,2
무슨 날인가,"5,326","5,677",5,3
기온,"7,213","4,601",2,4
몇 월인가,"5,423","2,262",4,5


[표 2] [표 1]의 「실제 LNG로 본 폭」이 나온 계산 — 두 해의 시각을 그 변수의 값이 같은 것끼리 묶고, 묶음마다 실제 LNG를 평균 내서, 평균이 가장 많은 묶음 − 가장 적은 묶음


,어떻게 묶었나,실제 LNG 평균이 가장 많은 묶음 (그 묶음의 시각 수),실제 LNG 평균이 가장 적은 묶음 (그 묶음의 시각 수),폭 = 가장 많은 묶음의 평균 − 가장 적은 묶음의 평균
변수,,,,
몇 시인가,0 ~ 23시마다 — 24묶음,18시 (시각 699개),3시 (시각 699개),"22,396 − 12,614 = 9,782"
햇빛(일사) — 낮 10 ~ 15시만,낮 시각의 흔한 범위(0.21 ~ 3.28 MJ/m²)를 같은 너비의 열다섯 구간으...,0.21 ~ 0.41 MJ/m² (시각 281개),3.08 ~ 3.28 MJ/m² (시각 184개),"22,427 − 16,393 = 6,034"
무슨 날인가,날 종류(월 · 화~금 · 토 · 일 · 공휴일 · 명절)마다 — 6묶음,"화~금 (시각 9,096개)","일 (시각 2,370개)","19,446 − 14,120 = 5,326"
기온,흔한 범위(-3.2 ~ 29.1도) 안에서 반올림해 1도마다 — 33묶음,29도 (시각 212개),14도 (시각 512개),"22,144 − 14,931 = 7,213"
몇 월인가,1 ~ 12월마다 — 12묶음,"1월 (시각 1,485개)","10월 (시각 1,488개)","20,648 − 15,225 = 5,423"


[표 3] [표 2]의 계산을 몇 월인가로 열두 묶음 모두 — 달마다 LNG 평균 (한 시각의 LNG, MWh · 2023 · 2024년)
        「모델이 맞힌 평균」은 두 해의 모든 시각에서 달만 그 달로 바꿔 넣고 그래디언트 부스팅(물음 1에서 고른 모델)이 맞힌 LNG의 평균이다.
        예를 들어 7월 14시 시각이면 기온 · 햇빛 · 몇 시 · 무슨 날은 그 시각의 값 그대로 두고 달만 1월로 넣는다.
        열두 달 모두 같은 날씨 · 같은 시각으로 맞히니, 달끼리의 차이는 달 하나만 바꾼 차이다.


,그 달의 시각 수,그 달의 평균 기온 (도),실제 LNG의 평균 (모델 없이),모델이 맞힌 평균 (그래디언트 부스팅 · 모든 시각을 그 달로)
달,,,,
1월,"1,485",-1.0,"20,648","17,672"
2월,"1,344",3.0,"19,035","17,570"
3월,"1,485",8.4,"18,933","19,007"
4월,"1,440",15.0,"16,115","17,598"
5월,"1,488",19.0,"15,726","17,456"
6월,"1,440",24.0,"16,956","17,504"
7월,"1,488",26.6,"19,203","17,752"
8월,744,27.2,"19,229","17,407"
9월,"1,437",24.6,"18,090","17,627"


실제 LNG — 가장 많은 1월 20,648 − 가장 적은 10월 15,225 = 5,423 ([표 1]의 몇 월인가 「실제 LNG로 본 폭」과 같다)
모델이 맞힌 값 — 가장 많은 3월 19,007 − 가장 적은 11월 16,903 = 2,104 (2-나에서 그래디언트 부스팅이 낸 몇 월인가 폭과 같다. [표 1]의 2,262은 모델 넷이 낸 폭의 평균이라 다르다)


## 2-사 — 기온 · 햇빛이 한 칸 움직이면 LNG가 얼마나 움직이나 (1도 · 1 MJ/m²에 몇 MWh, 모델 넷)

**어떻게 내나.** 1-마 [표 2]와 같은 계산입니다. 배운 두 해(2023 · 2024년)의 시각을 그대로 두고 기온이나 햇빛 하나만 바꿔 모델 넷으로 다시 맞힌 뒤, 맞힌 LNG의 평균이 움직인 양을 바꾼 크기로 나눕니다. 바꾸는 폭은 1-마와 같습니다 — 기온 추운 쪽은 16도에서 흔한 범위의 추운 쪽 끝(영하 3.2도)까지, 더운 쪽은 24도에서 더운 쪽 끝(29.1도)까지, 햇빛은 낮 10 ~ 15시에 일사가 약한 끝(0.21)에서 강한 끝(3.28 MJ/m²)까지입니다. 몇 시인가 · 무슨 날인가 · 몇 월인가는 한 칸이 크기를 갖지 않는 범주라 넣지 않았습니다 — 1-마 [표 2]의 「3시 → 18시」 · 「수요일 → 일요일」이 그 대신입니다. 2-나에서 만든 모델과 바꾸는 함수를 쓰니, 준비 셀 셋 → 2-나 → 2-사 순서로 돌립니다. [표 2]의 바꿔 넣어 보기 폭은 2-나와 같은 계산을 이 셀에서 다시 합니다. 2025년 LNG는 쓰지 않습니다.

**표의 숫자** — [표 1]의 칸은 한 칸(1도 · 1 MJ/m²) 움직일 때 모델이 맞힌 LNG가 움직인 양입니다(MWh, 한 시각 평균). +면 LNG가 늘고, −면 줍니다. 줄마다 한 칸의 단위가 달라서 줄끼리 크기를 비교하는 표가 아닙니다. [표 2]는 [표 1]의 넷의 평균에 흔히 움직이는 폭을 곱한 값을, 바꿔 넣어 보기(2-나)에서 비중을 비교할 때 쓴 폭과 나란히 놓았습니다. 곱한 값은 + · −를 떼고 크기만 적었습니다 — 바꿔 넣어 보기의 폭은 「가장 많이 맞힌 값 − 가장 적게 맞힌 값」이라 늘 +이기 때문입니다.

- [표 1]의 + · −는 1-마에서 「같다」였던 쪽과 같아야 합니다 — 추울수록 · 더울수록 +, 햇빛이 강할수록 −.
- [표 2]에서 곱한 값과 바꿔 넣어 보기의 폭이 거의 같으면, 「1도에 얼마」에 흔히 움직이는 폭을 곱한 것이 비중을 비교할 때 쓴 크기라는 뜻입니다. 기온은 바꿔 넣어 보기에서 가장 적게 맞힌 기온이 꼭 16도가 아니라서 조금 다를 수 있습니다.
- [표 1]에서 햇빛 줄과 기온 줄이 몇 배 차이 나는지와 [표 2]의 곱한 값끼리 몇 배 차이 나는지가 많이 다르면, 단위가 붙은 숫자로는 변수끼리 비교할 수 없다는 뜻입니다.
- 결론 칸과 인과지도에는 넷의 평균을 적습니다(2-가 — 한 모델로 정하지 않는다).

In [107]:
# 8단계 모델 확정과 핵심인자 · 2-사
assert "평균예측" in globals() and "틀" in globals(), "2-나를 먼저 돌려 주세요 — 모델 넷과 「한 가지만 바꿔 맞히기」가 2-나에서 만들어집니다"
낮시각 = 틀["hour24"].between(10, 15)                                             # 낮 10 ~ 15시 — 2-라가 「낮」 · 「폭」 같은 이름을 다시 써서, 이 셀에서 쓸 것은 다른 이름으로 다시 만든다
기온범위 = 틀["ta"].quantile([0.05, 0.95])                                         # 흔한 범위 — 배운 두 해 시각의 양 끝 5%씩을 뺀다(1-마 · 2-나와 같다)
일사범위 = 틀.loc[낮시각, "icsr"].quantile([0.05, 0.95])                           # 낮 시각의 일사도 같은 방법으로
한칸 = {                                                                          # 줄 이름: (바꾸기 전, 바꾼 뒤, 바꾼 크기, 한 칸의 단위) — 1-마 [표 2]와 같은 바꾸기
    "기온 — 16도보다 1도 추울 때마다": (기온을(틀, 16), 기온을(틀, 기온범위[0.05]), 16 - 기온범위[0.05], "도"),
    "기온 — 24도보다 1도 더울 때마다": (기온을(틀, 24), 기온을(틀, 기온범위[0.95]), 기온범위[0.95] - 24, "도"),
    "햇빛 — 낮에 일사가 1 MJ/m² 강할 때마다": (일사를(틀[낮시각], 일사범위[0.05]), 일사를(틀[낮시각], 일사범위[0.95]), 일사범위[0.95] - 일사범위[0.05], "MJ/m²"),
}
한칸에 = {}
for 이름, (앞d, 뒤d, 칸, _) in 한칸.items():
    앞, 뒤 = 평균예측(앞d), 평균예측(뒤d)                                           # 바꾸기 전 · 바꾼 뒤 맞힌 LNG의 평균 — 1-마와 같은 계산
    한칸에[이름] = pd.Series({m: (뒤[m] - 앞[m]) / 칸 for m in 모델들})             # 움직인 양을 바꾼 크기로 나눈다 = 한 칸에 몇 MWh
표1 = pd.DataFrame(한칸에).T                                                      # 줄: 한 칸, 칸: 모델
표1["넷의 평균"] = 표1.mean(axis=1)                                                # 2-가 — 한 모델로 정하지 않는다

print("[표 1] 한 칸 움직일 때 모델이 맞힌 LNG가 움직인 양 (MWh, 한 시각 평균) — +면 LNG가 는다 · −면 준다")
print("        1-마 [표 2]의 움직인 양을 바꾼 크기로 나눈 값 · 줄마다 한 칸의 단위(도 · MJ/m²)가 달라 줄끼리 비교하는 표가 아니다")
display(표1.apply(lambda c: c.map(lambda v: f"{v:+,.0f}")).rename_axis("한 칸"))

훑을값 = {"기온 — 16도보다 1도 추울 때마다": (틀, np.unique(np.r_[기온범위[0.05], np.arange(np.ceil(기온범위[0.05]), np.floor(기온범위[0.95]) + 1), 기온범위[0.95]]), 기온을),
          "햇빛 — 낮에 일사가 1 MJ/m² 강할 때마다": (틀[낮시각], np.linspace(일사범위[0.05], 일사범위[0.95], 16), 일사를)}   # 2-나와 같은 값들(기온은 1도마다, 햇빛은 열여섯 값)
나폭 = {}
for 줄, (d, 값들, 바꾸기) in 훑을값.items():
    맞힘 = pd.DataFrame({v: 평균예측(바꾸기(d, v)) for v in 값들})               # 줄: 모델, 칸: 넣어 본 값 — 2-나와 같은 계산을 이 셀에서 다시 한다
    나폭[줄] = (맞힘.max(axis=1) - 맞힘.min(axis=1)).mean()                     # 가장 많이 맞힌 값 − 가장 적게 맞힌 값, 모델 넷의 평균 = 2-나 [표 1] 「넷의 평균」
표2 = pd.DataFrame({줄: {"한 칸에 얼마 (넷의 평균)": f"{표1.loc[줄, '넷의 평균']:+,.0f}",
                        "흔히 움직이는 폭": f"{한칸[줄][2]:.1f}도" if 한칸[줄][3] == "도" else f"{한칸[줄][2]:.2f} MJ/m²",
                        "곱한 값 (+ · −를 뗀 크기)": f"{abs(표1.loc[줄, '넷의 평균'] * 한칸[줄][2]):,.0f}",   # 한 칸에 얼마 × 흔히 움직이는 폭
                        "바꿔 넣어 보기(2-나)의 폭 (넷의 평균)": f"{나폭[줄]:,.0f}"}                       # 비중을 비교할 때 쓴 폭
                    for 줄 in 훑을값}).T
print("[표 2] 한 칸에 얼마 × 흔히 움직이는 폭 = 곱한 값 — 바꿔 넣어 보기(2-나)의 폭과 나란히 (MWh, 한 시각 평균)")
display(표2.rename_axis("한 칸"))

[표 1] 한 칸 움직일 때 모델이 맞힌 LNG가 움직인 양 (MWh, 한 시각 평균) — +면 LNG가 는다 · −면 준다
        1-마 [표 2]의 움직인 양을 바꾼 크기로 나눈 값 · 줄마다 한 칸의 단위(도 · MJ/m²)가 달라 줄끼리 비교하는 표가 아니다


,선형회귀,기온 × 시간대 선형회귀,랜덤포레스트,그래디언트 부스팅,넷의 평균
한 칸,,,,,
기온 — 16도보다 1도 추울 때마다,+281,+263,+197,+214,+239
기온 — 24도보다 1도 더울 때마다,+977,+587,+545,+534,+661
햇빛 — 낮에 일사가 1 MJ/m² 강할 때마다,"-2,604","-2,435","-2,029","-2,248","-2,329"


[표 2] 한 칸에 얼마 × 흔히 움직이는 폭 = 곱한 값 — 바꿔 넣어 보기(2-나)의 폭과 나란히 (MWh, 한 시각 평균)


,한 칸에 얼마 (넷의 평균),흔히 움직이는 폭,곱한 값 (+ · −를 뗀 크기),바꿔 넣어 보기(2-나)의 폭 (넷의 평균)
한 칸,,,,
기온 — 16도보다 1도 추울 때마다,+239,19.2도,"4,584","4,601"
햇빛 — 낮에 일사가 1 MJ/m² 강할 때마다,"-2,329",3.07 MJ/m²,"7,150","7,150"


# 물음 3 — 1단계에 짐작으로 적은 원인 여섯 줄이 맞았나

> **결정** — 여섯 줄 가운데 맞은 것은 「추우면 LNG가 는다」 하나이고, 「발전용이면 계절 영향이 난방만큼 크지 않을 것」은 반대였다. 나머지 넷(탄소 · 연료별 경제성 · 지정학 위험 · 상대 가격)은 모델 밖에 두어 여기서는 볼 수 없다